In [1]:
import cv2
import os
import numpy as np
import time

def import_frames(folder_path):
    image_list = []
    
    # Ensure the folder path is valid
    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' does not exist.")
        return image_list
    
    # Get a sorted list of filenames in the folder
    sorted_filenames = sorted(os.listdir(folder_path))
    
    # Iterate through sorted files in the folder
    for filename in sorted_filenames:
        file_path = os.path.join(folder_path, filename)
        
        # Check if the file is an image (you can add more image extensions if needed)
        if file_path.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            # Read the image
     
            img = cv2.imread(file_path)
            
            # Append the image to the list
            if img is not None:
                gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                image_list.append(gray_img)
            else:
                print(f"Error reading image: {filename}")
    
    return image_list



def create_video(images_lists, text_list, output_path, fps=24):
    # Get the height and width of the frames
    height, width = images_lists[0][0].shape[:2]

    # Define the codec and create a VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (2 * width, 2 * height), isColor=False)

    # Iterate through the frames in each list
    for i in range(len(images_lists[0])):
        # Create a 2 by 2 grid by concatenating images horizontally and vertically
        top_row = np.concatenate((images_lists[0][i], images_lists[1][i]), axis=1)
        bottom_row = np.concatenate((images_lists[2][i], images_lists[3][i]), axis=1)
        final_frame = np.concatenate((top_row, bottom_row), axis=0)
        
       
        # Add text to each space in the grid
        for j, text in enumerate(text_list):
            text_position = (width * (j % 2), height * (j // 2) + 20)
            cv2.putText(final_frame, str(text), text_position, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
 
        # Write the frame to the video file
        video_writer.write(final_frame)

    # Release the VideoWriter object
    video_writer.release()


def dice_coef(groundtruth_mask, pred_mask):
    intersect = np.sum(pred_mask*groundtruth_mask)
    total_sum = np.sum(pred_mask) + np.sum(groundtruth_mask)
    dice = np.mean(2*intersect/total_sum)
    if np.isnan(dice): # Ground truth and pred all zeros
        dice = 1.0
    return round(dice, 3) 

def iou_coef(groundtruth_mask, pred_mask):
    intersect = np.sum(pred_mask*groundtruth_mask)
    union = np.sum(pred_mask) + np.sum(groundtruth_mask) - intersect
    iou = np.mean(intersect/union)
    if np.isnan(iou): # Ground truth and pred all zeros
        iou = 1.0
    return round(iou, 3)

def precision_score(groundtruth_mask, pred_mask):
    intersect = np.sum(pred_mask*groundtruth_mask)
    total_pixel_pred = np.sum(pred_mask)
    precision = np.mean(intersect/total_pixel_pred)
    if np.isnan(precision):
        precision = 1.0
    return round(precision, 3)

def accuracy_score(groundtruth_mask, pred_mask):
    intersect = np.sum(pred_mask*groundtruth_mask)
    union = np.sum(pred_mask) + np.sum(groundtruth_mask) - intersect
    xor = np.sum(groundtruth_mask==pred_mask)
    acc = np.mean(xor/(union + xor - intersect))
    return round(acc, 3)



def get_background(first_frames):
    # Return an image that is the average of the frames, first_frames is a list
    return np.mean(first_frames, axis=0).astype(np.uint8)

def threshold_background(frame, background):
    # Perform background subtraction
    diff = cv2.absdiff(frame, background)
    _, movement = cv2.threshold(diff,20, 255,cv2.THRESH_BINARY)
    return movement

def background_subtraction(input_sequence, num_frames):

    initial_frames = input_sequence[:num_frames]
    mean_background = get_background(initial_frames)
    background_subtraction_sequence = [np.zeros_like(input_sequence[0])] * num_frames

    for i in range(num_frames, len(input_sequence)):
        background_subtracted = threshold_background(input_sequence[i], mean_background)
        background_subtraction_sequence.append(background_subtracted)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    return background_subtraction_sequence
    

# Import input sequences
wdir = 'D:/IMCV/2nd semester/VR-Visual Recognition/Practical/Lab 2'
input_sequence_path = os.path.join(wdir, 'pedestrians', 'input') 
input_sequence = import_frames(input_sequence_path) 

# Import ground truth sequences
groundtruth_sequence_path = os.path.join(wdir, 'pedestrians', 'groundtruth') 
groundtruth_sequence = import_frames(groundtruth_sequence_path)

# Create background subtractors
bs_mog2 = cv2.createBackgroundSubtractorMOG2()
bs_knn = cv2.createBackgroundSubtractorKNN()

# Apply background subtraction to each frame in the sequences
start_mog = time.time()
background_sub_sequence_mog = [bs_mog2.apply(frame) for frame in input_sequence]
time_mog = round(time.time() - start_mog, 3)
# Convert to binary
background_subtraction_sequence_mog = []
for gt_frame in background_sub_sequence_mog:
    _, gt_bin = cv2.threshold(gt_frame, 30, 1, cv2.THRESH_BINARY)
    background_subtraction_sequence_mog.append(gt_bin)

start_knn = time.time()
background_sub_sequence_knn = [bs_knn.apply(frame) for frame in input_sequence]
time_knn = round(time.time() - start_knn, 3)

background_subtraction_sequence_knn = []
for gt_frame in background_sub_sequence_knn:
    _, gt_bin = cv2.threshold(gt_frame, 30, 1, cv2.THRESH_BINARY)
    background_subtraction_sequence_knn.append(gt_bin)

start_bs = time.time()
background_sub_sequence = background_subtraction(input_sequence, 60)
time_bs = round(time.time() - start_bs, 3)

background_subtraction_sequence = []
for gt_frame in background_sub_sequence:
    _, gt_bin = cv2.threshold(gt_frame, 30, 1, cv2.THRESH_BINARY)
    background_subtraction_sequence.append(gt_bin)

output_video_path = os.path.join(wdir, 'output_video.avi')
text_list = [f"BS - MOG, Comp time: {time_mog}", f"BS - KNN, Comp time: {time_knn}",
             f"BS - Comp time: {time_bs}", "Groundtruth"]

sequences = [background_sub_sequence_mog, background_sub_sequence_knn,
             background_sub_sequence, groundtruth_sequence]


create_video(sequences, text_list, output_video_path, fps=24)
print(f"Video saved to: {output_video_path}")


# Convert ground truth sequence to binary
grt_sequence = []
for gt_frame in groundtruth_sequence:
    _, gt_bin = cv2.threshold(gt_frame, 30, 1, cv2.THRESH_BINARY)
    grt_sequence.append(gt_bin)
    

# Metrics - BS Mog
dice_tot = []
precision_tot = []
accuracy_tot = []
iou_tot = []

for gt_frame, bs_frame in zip(grt_sequence[400:], background_subtraction_sequence_mog[400:]):
    
    dice = dice_coef(gt_frame, bs_frame)
    precision = precision_score(gt_frame, bs_frame)
    accuracy  = accuracy_score(gt_frame, bs_frame)
    iou = iou_coef(gt_frame, bs_frame)
    
    dice_tot.append(dice)
    precision_tot.append(precision)
    accuracy_tot.append(accuracy)
    iou_tot.append(iou)
    
print(f"BS Mog - Efficiency: {time_mog}")
print(f"BS Mog - Dice Coefficient: {round(np.mean(dice_tot),3)}")
print(f"BS Mog - Pixel Accuracy: {round(np.mean(accuracy_tot),3)}")
print(f"BS Mog - Pixel Precision: {round(np.mean(precision_tot),3)}")
print(f"BS Mog - IoU Metric: {round(np.mean(iou_tot),3)}")
      
      

# Metrics - KNN
dice_tot = []
precision_tot = []
accuracy_tot = []
iou_tot = []


for gt_frame, bs_frame in zip(grt_sequence[400:], background_subtraction_sequence_knn[400:]):
    dice = dice_coef(gt_frame, bs_frame)
    precision = precision_score(gt_frame, bs_frame)
    accuracy  = accuracy_score(gt_frame, bs_frame)
    iou = iou_coef(gt_frame, bs_frame)
    
    dice_tot.append(dice)
    precision_tot.append(precision)
    accuracy_tot.append(accuracy)
    iou_tot.append(iou)
      
print(f"BS KNN - Efficiency: {time_knn}")
print(f"BS KNN - Pixel Accuracy: {round(np.mean(accuracy_tot),3)}")
print(f"BS KNN - Pixel Precision: {round(np.mean(precision_tot),3)}")
print(f"BS KNN - Dice Coefficient: {round(np.mean(dice_tot),3)}")
print(f"BS KNN - IoU Metric: {round(np.mean(iou_tot),3)}")




# Metrics - BS Mean
dice_tot = []
precision_tot = []
accuracy_tot = []
iou_tot = []


for gt_frame, bs_frame in zip(grt_sequence[400:], background_subtraction_sequence[400:]):
    
    dice = dice_coef(gt_frame, bs_frame)
    precision = precision_score(gt_frame, bs_frame)
    accuracy  = accuracy_score(gt_frame, bs_frame)
    iou = iou_coef(gt_frame, bs_frame)
    
    dice_tot.append(dice)
    precision_tot.append(precision)
    accuracy_tot.append(accuracy)
    iou_tot.append(iou)
      
print(f"BS Mean - Efficiency: {time_bs}")
print(f"BS Mean - Pixel Accuracy: {round(np.mean(accuracy_tot),3)}")
print(f"BS Mean - Pixel Precision: {round(np.mean(precision_tot),3)}")
print(f"BS Mean - Dice Coefficient: {round(np.mean(dice_tot),3)}")
print(f"BS Mean - IoU Metric: {round(np.mean(iou_tot),3)}")



Video saved to: D:/IMCV/2nd semester/VR-Visual Recognition/Practical/Lab 2\output_video.avi
BS Mog - Efficiency: 2.083
BS Mog - Dice Coefficient: 0.617
BS Mog - Pixel Accuracy: 0.991
BS Mog - Pixel Precision: 0.621
BS Mog - IoU Metric: 0.521


C:\Users\SOFIA\Anaconda3\lib\site-packages\ipykernel_launcher.py:69: RuntimeWarning: invalid value encountered in longlong_scalars
C:\Users\SOFIA\Anaconda3\lib\site-packages\ipykernel_launcher.py:85: RuntimeWarning: invalid value encountered in ulong_scalars
C:\Users\SOFIA\Anaconda3\lib\site-packages\ipykernel_launcher.py:77: RuntimeWarning: invalid value encountered in ulong_scalars


BS KNN - Efficiency: 2.562
BS KNN - Pixel Accuracy: 0.993
BS KNN - Pixel Precision: 0.828
BS KNN - Dice Coefficient: 0.716
BS KNN - IoU Metric: 0.622
BS Mean - Efficiency: 16.239
BS Mean - Pixel Accuracy: 0.967
BS Mean - Pixel Precision: 0.343
BS Mean - Dice Coefficient: 0.413
BS Mean - IoU Metric: 0.289
